# BC Parks — Exploratory Analysis

Starter notebook for the MDS capstone. Loads the four datasets fetched by `python -m src.download` and walks through:

1. Loading & summary stats
2. Parks by type / class
3. Map of every protected area
4. Advisories: event types, urgency, duration
5. Map of currently active advisories
6. Activities & facilities per park

If a dataset is missing, run `python -m src.download` from the project root first.

In [1]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px

DATA = Path('..') / 'data'


def load(name: str) -> pd.DataFrame:
    return pd.json_normalize(json.loads((DATA / f'{name}.json').read_text()))


parks      = load('parks')
advisories = load('advisories')
activities = load('activities')
facilities = load('facilities')

print(f'parks      : {len(parks):>5}')
print(f'advisories : {len(advisories):>5}')
print(f'activities : {len(activities):>5}')
print(f'facilities : {len(facilities):>5}')
parks.head(3)

parks      :  1052
advisories :   474
activities :  4462
facilities :  1869


,documentId,orcs,protectedAreaName,type,class,typeCode,legalStatus,latitude,longitude,totalArea,uplandArea,marineArea,establishedDate,url,slug,isDisplayed,hasCampfireBan,description,safetyInfo,locationNotes
0,vasrda40sqc7xgua7ir1sio6,82,Loon Lake Park,Park,A,PK,Active,51.104886,-121.257278,6.0,6.0,0.0,1956-03-23,https://bcparks.ca/loon-lake-park/,loon-lake-park,True,False,<p>\n All facilities have been removed and Lo...,,
1,wjncf68dyd6exlya9pn3vt9k,180,Wood Mountain Ski Park,Park,C,PK,Active,49.660321,-125.165752,97.0,97.0,0.0,1966-04-26,https://bcparks.ca/wood-mountain-ski-park/,wood-mountain-ski-park,True,False,,,
2,meyeyycjvu285afmnfmrmzsn,486,Lower Skeena River Park,Park,A,PK,Active,54.299823,-129.341533,582.0,582.0,0.0,2004-05-17,https://bcparks.ca/lower-skeena-river-park/,lower-skeena-river-park,True,False,<p>This park was established as a result of th...,,


## 1. Parks by type and legal status

In [2]:
by_type = (
    parks.groupby('type', dropna=False).size().reset_index(name='count')
    .sort_values('count', ascending=False)
)
fig = px.bar(by_type, x='type', y='count', title='Protected areas by type')
fig.update_layout(xaxis_tickangle=-30, height=420)
fig.show()

parks['legalStatus'].value_counts(dropna=False)

legalStatus
Active      1049
Repealed       3
Name: count, dtype: int64

## 2. Map of every protected area

In [3]:
geo = parks.dropna(subset=['latitude', 'longitude']).copy()
geo['totalArea'] = geo['totalArea'].fillna(0).clip(lower=0)

fig = px.scatter_geo(
    geo,
    lat='latitude',
    lon='longitude',
    size=geo['totalArea'].pow(0.5) + 1,
    color='type',
    hover_name='protectedAreaName',
    hover_data={'orcs': True, 'totalArea': ':.0f', 'type': False, 'latitude': False, 'longitude': False},
    scope='north america',
    title=f'BC protected areas (n={len(geo)})',
)
fig.update_geos(fitbounds='locations', showsubunits=True, subunitcolor='#888')
fig.update_layout(height=600, legend_title_text='Type')
fig.show()

## 3. Advisories — types, urgency, and how long they last

In [4]:
adv = advisories.copy()
for col in ['advisoryDate', 'effectiveDate', 'endDate', 'expiryDate', 'updatedDate']:
    if col in adv.columns:
        adv[col] = pd.to_datetime(adv[col], errors='coerce', utc=True)

adv['duration_days'] = (adv['endDate'] - adv['effectiveDate']).dt.total_seconds() / 86400

by_event = adv['eventType.eventType'].value_counts().reset_index()
by_event.columns = ['eventType', 'count']
fig = px.bar(by_event.head(15), x='eventType', y='count', title='Top advisory event types')
fig.update_layout(xaxis_tickangle=-30, height=420)
fig.show()

print('By urgency:')
print(adv['urgency.urgency'].value_counts(dropna=False))
print('\nBy status:')
print(adv['advisoryStatus.advisoryStatus'].value_counts(dropna=False))

By urgency:
urgency.urgency
Low       304
Medium    128
High       42
Name: count, dtype: int64

By status:
advisoryStatus.advisoryStatus
Published    474
Name: count, dtype: int64


## 4. Currently active advisories on a map

In [5]:
now = pd.Timestamp.now(tz='UTC')
active = adv[
    adv['advisoryStatus.advisoryStatus'].eq('Published')
    & adv['effectiveDate'].le(now)
    & (adv['endDate'].isna() | adv['endDate'].ge(now))
].copy()

park_coords = parks.set_index('orcs')[['latitude', 'longitude']].rename(
    columns={'latitude': 'p_lat', 'longitude': 'p_lon'}
)


def first_orcs(row):
    nodes = row.get('protectedAreas_connection.nodes')
    if isinstance(nodes, list) and nodes:
        return nodes[0].get('orcs')
    return None


active['orcs'] = active.apply(first_orcs, axis=1)
active = active.merge(park_coords, left_on='orcs', right_index=True, how='left')
active['lat'] = active['latitude'].fillna(active['p_lat'])
active['lon'] = active['longitude'].fillna(active['p_lon'])
active_geo = active.dropna(subset=['lat', 'lon'])

fig = px.scatter_geo(
    active_geo,
    lat='lat',
    lon='lon',
    color='eventType.eventType',
    hover_name='title',
    hover_data={'urgency.urgency': True, 'effectiveDate': True, 'endDate': True,
                'lat': False, 'lon': False, 'eventType.eventType': False},
    scope='north america',
    title=f'Active advisories (n={len(active_geo)})',
)
fig.update_geos(fitbounds='locations', showsubunits=True)
fig.update_layout(height=600, legend_title_text='Event')
fig.show()

## 5. Activities & facilities per park

In [6]:
act_open = activities[activities['isActive'].fillna(False) & activities['isActivityOpen'].fillna(False)]
top_parks = (
    act_open.groupby(['protectedArea.orcs', 'protectedArea.protectedAreaName'])
    .size()
    .reset_index(name='open_activities')
    .sort_values('open_activities', ascending=False)
    .head(20)
)
fig = px.bar(
    top_parks,
    x='open_activities',
    y='protectedArea.protectedAreaName',
    orientation='h',
    title='Top 20 parks by open activities',
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=600)
fig.show()

popular = (
    act_open['activityType.activityName']
    .value_counts()
    .head(15)
    .rename_axis('activity')
    .reset_index(name='parks_offering')
)
fig = px.bar(popular, x='activity', y='parks_offering', title='Most-offered activities (open)')
fig.update_layout(xaxis_tickangle=-30, height=420)
fig.show()

/var/folders/7b/0ypr4cms2fj3pnlz5m8_6tlw0000gn/T/ipykernel_2556/2954182324.py:1: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



In [7]:
fac_open = facilities[facilities['isActive'].fillna(False) & facilities['isFacilityOpen'].fillna(False)]
fac_top = (
    fac_open['facilityType.facilityName']
    .value_counts()
    .head(15)
    .rename_axis('facility')
    .reset_index(name='parks_offering')
)
fig = px.bar(fac_top, x='facility', y='parks_offering', title='Most common open facilities')
fig.update_layout(xaxis_tickangle=-30, height=420)
fig.show()

/var/folders/7b/0ypr4cms2fj3pnlz5m8_6tlw0000gn/T/ipykernel_2556/976107318.py:1: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

